In [2]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 200)


# Load

In [3]:
app_df = pd.read_csv("https://raw.githubusercontent.com/arawsardni/5---Gaung-Taqwa-Indraswara---Sandra-Triana-Nursyafri/refs/heads/main/Dataset/raw/application_record.csv")
cred_df = pd.read_csv("https://raw.githubusercontent.com/arawsardni/5---Gaung-Taqwa-Indraswara---Sandra-Triana-Nursyafri/refs/heads/main/Dataset/raw/credit_record.csv")

In [4]:
app_df

,ID,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,DAYS_BIRTH,DAYS_EMPLOYED,FLAG_MOBIL,FLAG_WORK_PHONE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,CNT_FAM_MEMBERS
0,5008804,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,-12005,-4542,1,1,0,0,NaN,2.0
1,5008805,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,-12005,-4542,1,1,0,0,NaN,2.0
2,5008806,M,Y,Y,0,112500.0,Working,Secondary / secondary special,Married,House / apartment,-21474,-1134,1,0,0,0,Security staff,2.0
3,5008808,F,N,Y,0,270000.0,Commercial associate,Secondary / secondary special,Single / not married,House / apartment,-19110,-3051,1,0,1,1,Sales staff,1.0
4,5008809,F,N,Y,0,270000.0,Commercial associate,Secondary / secondary special,Single / not married,House / apartment,-19110,-3051,1,0,1,1,Sales staff,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
438552,6840104,M,N,Y,0,135000.0,Pensioner,Secondary / secondary special,Separated,House / apartment,-22717,365243,1,0,0,0,NaN,1.0
438553,6840222,F,N,N,0,103500.0,Working,Secondary / secondary special,Single / not married,House / apartment,-15939,-3007,1,0,0,0,Laborers,1.0
438554,6841878,F,N,N,0,54000.0,Commercial associate,Higher education,Single / not married,With parents,-8169,-372,1,1,0,0,Sales staff,1.0
438555,6842765,F,N,Y,0,72000.0,Pensioner,Secondary / secondary special,Married,House / apartment,-21673,365243,1,0,0,0,NaN,2.0


In [5]:
cred_df

,ID,MONTHS_BALANCE,STATUS
0,5001711,0,X
1,5001711,-1,0
2,5001711,-2,0
3,5001711,-3,0
4,5001712,0,C
...,...,...,...
1048570,5150487,-25,C
1048571,5150487,-26,C
1048572,5150487,-27,C
1048573,5150487,-28,C


In [13]:
cred_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1048575 entries, 0 to 1048574
Data columns (total 3 columns):
 #   Column          Non-Null Count    Dtype 
---  ------          --------------    ----- 
 0   ID              1048575 non-null  int64 
 1   MONTHS_BALANCE  1048575 non-null  int64 
 2   STATUS          1048575 non-null  object
dtypes: int64(2), object(1)
memory usage: 24.0+ MB


# Creating Target

In [44]:
is_bad = (
    cred_df['STATUS']
    .isin(['1', '2', '3','4','5'])
    .groupby(cred_df['ID'])
    .max()  # Jika ada minimal 1 True, return 1
    .astype(int)
)

# Aggregrating Credit Records

In [45]:
credit_agg = cred_df.groupby('ID').agg(
    CREDIT_HISTORY_LENGTH=('MONTHS_BALANCE', lambda x: abs(x.min() - x.max()) + 1),  # Panjang riwayat kredit
    BAD_DEBT_RATIO=('STATUS', lambda x: (x.isin(['1', '2', '3', '4', '5']).sum()) / len(x)),  # Proporsi keterlambatan >30 hari
    AVERAGE_DELAYED_MONTHS=('STATUS', lambda x: x[x.isin(['1', '2', '3', '4', '5'])].astype(int).mean() if x.isin(['1', '2', '3', '4', '5']).any() else 0),
).reset_index()

# Gabungkan target dengan fitur agregasi kredit
credit_agg['IS_HIGH_RISK'] = is_bad.values

In [46]:
credit_agg

,ID,CREDIT_HISTORY_LENGTH,BAD_DEBT_RATIO,AVERAGE_DELAYED_MONTHS,IS_HIGH_RISK
0,5001711,4,0.0,0.0,0
1,5001712,19,0.0,0.0,0
2,5001713,22,0.0,0.0,0
3,5001714,15,0.0,0.0,0
4,5001715,60,0.0,0.0,0
...,...,...,...,...,...
45980,5150482,18,0.0,0.0,0
45981,5150483,18,0.0,0.0,0
45982,5150484,13,0.0,0.0,0
45983,5150485,2,0.0,0.0,0


In [29]:
cred_df[cred_df["ID"] == 5001712]

,ID,MONTHS_BALANCE,STATUS
4,5001712,0,C
5,5001712,-1,C
6,5001712,-2,C
7,5001712,-3,C
8,5001712,-4,C
9,5001712,-5,C
10,5001712,-6,C
11,5001712,-7,C
12,5001712,-8,C
13,5001712,-9,0


In [41]:
cred_df[cred_df["ID"] == 5001718]

,ID,MONTHS_BALANCE,STATUS
142,5001718,0,C
143,5001718,-1,C
144,5001718,-2,C
145,5001718,-3,0
146,5001718,-4,0
147,5001718,-5,0
148,5001718,-6,0
149,5001718,-7,0
150,5001718,-8,1
151,5001718,-9,X


# Merge Data

In [30]:
new_df = pd.merge(app_df, credit_agg, on="ID", how="inner")

In [31]:
new_df

,ID,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,DAYS_BIRTH,DAYS_EMPLOYED,FLAG_MOBIL,FLAG_WORK_PHONE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,CNT_FAM_MEMBERS,CREDIT_HISTORY_LENGTH,BAD_DEBT_RATIO,AVERAGE_DELAYED_MONTHS,COUNT_LATE_PAYMENTS,TOTAL_PAID_MONTHS,MAX_LATE_STATUS,IS_HIGH_RISK
0,5008804,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,-12005,-4542,1,1,0,0,NaN,2.0,16,0.062500,1.000000,1,13,1,1
1,5008805,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,-12005,-4542,1,1,0,0,NaN,2.0,15,0.066667,1.000000,1,12,1,1
2,5008806,M,Y,Y,0,112500.0,Working,Secondary / secondary special,Married,House / apartment,-21474,-1134,1,0,0,0,Security staff,2.0,30,0.000000,0.000000,0,7,0,0
3,5008808,F,N,Y,0,270000.0,Commercial associate,Secondary / secondary special,Single / not married,House / apartment,-19110,-3051,1,0,1,1,Sales staff,1.0,5,0.000000,0.000000,0,0,0,0
4,5008809,F,N,Y,0,270000.0,Commercial associate,Secondary / secondary special,Single / not married,House / apartment,-19110,-3051,1,0,1,1,Sales staff,1.0,5,0.000000,0.000000,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
36452,5149828,M,Y,Y,0,315000.0,Working,Secondary / secondary special,Married,House / apartment,-17348,-2420,1,0,0,0,Managers,2.0,12,0.333333,4.750000,4,0,5,1
36453,5149834,F,N,Y,0,157500.0,Commercial associate,Higher education,Married,House / apartment,-12387,-1325,1,0,1,1,Medicine staff,2.0,24,0.750000,2.944444,18,5,5,1
36454,5149838,F,N,Y,0,157500.0,Pensioner,Higher education,Married,House / apartment,-12387,-1325,1,0,1,1,Medicine staff,2.0,33,0.545455,2.944444,18,14,5,1
36455,5150049,F,N,Y,0,283500.0,Working,Secondary / secondary special,Married,House / apartment,-17958,-655,1,0,0,0,Sales staff,2.0,10,0.200000,1.500000,2,0,2,1


In [32]:
new_df.isna().sum()

ID                            0
CODE_GENDER                   0
FLAG_OWN_CAR                  0
FLAG_OWN_REALTY               0
CNT_CHILDREN                  0
AMT_INCOME_TOTAL              0
NAME_INCOME_TYPE              0
NAME_EDUCATION_TYPE           0
NAME_FAMILY_STATUS            0
NAME_HOUSING_TYPE             0
DAYS_BIRTH                    0
DAYS_EMPLOYED                 0
FLAG_MOBIL                    0
FLAG_WORK_PHONE               0
FLAG_PHONE                    0
FLAG_EMAIL                    0
OCCUPATION_TYPE           11323
CNT_FAM_MEMBERS               0
CREDIT_HISTORY_LENGTH         0
BAD_DEBT_RATIO                0
AVERAGE_DELAYED_MONTHS        0
COUNT_LATE_PAYMENTS           0
TOTAL_PAID_MONTHS             0
MAX_LATE_STATUS               0
IS_HIGH_RISK                  0
dtype: int64

In [33]:
new_df = new_df.dropna()

In [34]:
new_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 25134 entries, 2 to 36456
Data columns (total 25 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   ID                      25134 non-null  int64  
 1   CODE_GENDER             25134 non-null  object 
 2   FLAG_OWN_CAR            25134 non-null  object 
 3   FLAG_OWN_REALTY         25134 non-null  object 
 4   CNT_CHILDREN            25134 non-null  int64  
 5   AMT_INCOME_TOTAL        25134 non-null  float64
 6   NAME_INCOME_TYPE        25134 non-null  object 
 7   NAME_EDUCATION_TYPE     25134 non-null  object 
 8   NAME_FAMILY_STATUS      25134 non-null  object 
 9   NAME_HOUSING_TYPE       25134 non-null  object 
 10  DAYS_BIRTH              25134 non-null  int64  
 11  DAYS_EMPLOYED           25134 non-null  int64  
 12  FLAG_MOBIL              25134 non-null  int64  
 13  FLAG_WORK_PHONE         25134 non-null  int64  
 14  FLAG_PHONE              25134 non-null  int

In [35]:
new_df["IS_HIGH_RISK"].value_counts(normalize=True)

IS_HIGH_RISK
0    0.877099
1    0.122901
Name: proportion, dtype: float64

In [47]:
new_df["IS_HIGH_RISK"].value_counts()

IS_HIGH_RISK
0    22045
1     3089
Name: count, dtype: int64

In [25]:
for column in new_df.select_dtypes(include='object'):
    unique_values = new_df[column].unique()
    print(f"Unique values in {column}: {unique_values}")

Unique values in CODE_GENDER: ['M' 'F']
Unique values in FLAG_OWN_CAR: ['Y' 'N']
Unique values in FLAG_OWN_REALTY: ['Y' 'N']
Unique values in NAME_INCOME_TYPE: ['Working' 'Commercial associate' 'State servant' 'Student' 'Pensioner']
Unique values in NAME_EDUCATION_TYPE: ['Secondary / secondary special' 'Higher education' 'Incomplete higher'
 'Lower secondary' 'Academic degree']
Unique values in NAME_FAMILY_STATUS: ['Married' 'Single / not married' 'Civil marriage' 'Separated' 'Widow']
Unique values in NAME_HOUSING_TYPE: ['House / apartment' 'Rented apartment' 'Municipal apartment'
 'With parents' 'Co-op apartment' 'Office apartment']
Unique values in OCCUPATION_TYPE: ['Security staff' 'Sales staff' 'Accountants' 'Laborers' 'Managers'
 'Drivers' 'Core staff' 'High skill tech staff' 'Cleaning staff'
 'Private service staff' 'Cooking staff' 'Low-skill Laborers'
 'Medicine staff' 'Secretaries' 'Waiters/barmen staff' 'HR staff'
 'Realty agents' 'IT staff']


In [48]:
new_df["FLAG_MOBIL"].value_counts()

FLAG_MOBIL
1    25134
Name: count, dtype: int64

In [49]:
new_df.drop(["FLAG_MOBIL"], axis=1)

,ID,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,DAYS_BIRTH,DAYS_EMPLOYED,FLAG_WORK_PHONE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,CNT_FAM_MEMBERS,CREDIT_HISTORY_LENGTH,BAD_DEBT_RATIO,AVERAGE_DELAYED_MONTHS,COUNT_LATE_PAYMENTS,TOTAL_PAID_MONTHS,MAX_LATE_STATUS,IS_HIGH_RISK
2,5008806,M,Y,Y,0,112500.0,Working,Secondary / secondary special,Married,House / apartment,-21474,-1134,0,0,0,Security staff,2.0,30,0.000000,0.000000,0,7,0,0
3,5008808,F,N,Y,0,270000.0,Commercial associate,Secondary / secondary special,Single / not married,House / apartment,-19110,-3051,0,1,1,Sales staff,1.0,5,0.000000,0.000000,0,0,0,0
4,5008809,F,N,Y,0,270000.0,Commercial associate,Secondary / secondary special,Single / not married,House / apartment,-19110,-3051,0,1,1,Sales staff,1.0,5,0.000000,0.000000,0,0,0,0
5,5008810,F,N,Y,0,270000.0,Commercial associate,Secondary / secondary special,Single / not married,House / apartment,-19110,-3051,0,1,1,Sales staff,1.0,27,0.000000,0.000000,0,15,0,0
6,5008811,F,N,Y,0,270000.0,Commercial associate,Secondary / secondary special,Single / not married,House / apartment,-19110,-3051,0,1,1,Sales staff,1.0,39,0.000000,0.000000,0,27,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
36452,5149828,M,Y,Y,0,315000.0,Working,Secondary / secondary special,Married,House / apartment,-17348,-2420,0,0,0,Managers,2.0,12,0.333333,4.750000,4,0,5,1
36453,5149834,F,N,Y,0,157500.0,Commercial associate,Higher education,Married,House / apartment,-12387,-1325,0,1,1,Medicine staff,2.0,24,0.750000,2.944444,18,5,5,1
36454,5149838,F,N,Y,0,157500.0,Pensioner,Higher education,Married,House / apartment,-12387,-1325,0,1,1,Medicine staff,2.0,33,0.545455,2.944444,18,14,5,1
36455,5150049,F,N,Y,0,283500.0,Working,Secondary / secondary special,Married,House / apartment,-17958,-655,0,0,0,Sales staff,2.0,10,0.200000,1.500000,2,0,2,1


In [ ]:
new_df.corr(include)

ValueError: could not convert string to float: 'M'

# Feature Engineering

- AGE : abs(Days Birth / 365.25)
- WORK_YEARS : abs(Days Worked / 365.25)
- ACCOUNT_AGE : Months Balance min 
- 

# Transform Data 

ID:
* Drop the feature

Gender:
* One hot encoding

Age:
* Min-max scaling
* Fix skewness
* Abs value and div 365.25

Marital status:
* One hot encoding

Family member count
* Fix outliers

Children count
* Fix outliers
* Drop feature

Dwelling type
* One hot encoding

Income
* Remove outliers
* Fix skewness
* Min-max scaling

Job title
* One hot encoding
* Impute missing values

Employment status:
* One hot encoding

Education level:
* Ordinal encoding

Employment length:
* Remove outliers
* Min-max scaling
* Abs value and div 365.25
* change days of employments of retirees to 0

Has a car:
* Change it numerical
* One-hot encoding

Has a property:
* Change it numerical
* One-hot encoding

Has a mobile phone:
* Drop feature

Has a work phone:
* One-hot encoding

Has a phone:
* One-hot encoding

Has an email:
* One-hot encoding

Account age:

Is high risk(Target):
* Change the data type to numerical
* balance the data with SMOTE